# BÁO CÁO PHÂN TÍCH THÔNG SỐ DÒNG GIAO THÔNG (NGÃ TƯ HÀNG XANH)
**Môn học**: Hệ thống Giao thông Thông minh (ITS)
**Dữ liệu**: Trích xuất từ mô phỏng Eclipse SUMO qua 6 kịch bản (.sumocfg)
---
Notebook này phục vụ **Task 6 (Phân tích thông số dòng giao thông)** bằng cách đọc trực tiếp các tệp dữ liệu XML từ thư mục `../data/` và trực quan hóa các biểu đồ so sánh KPIs.

In [ ]:
# 1. Khai báo các thư viện cần thiết
import os
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI']
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data') if 'notebooks' in os.getcwd() else os.path.join(os.getcwd(), 'data')
print(f"Thư mục dữ liệu: {DATA_DIR}")

## 2. Nạp dữ liệu các Kịch bản từ `data/`

In [ ]:
SCENARIOS = [
    ('offpeak', '1. Thấp điểm', '#2ecc71'),
    ('baseline', '2. Tiêu chuẩn', '#3498db'),
    ('peak', '3. Cao điểm', '#e67e22'),
    ('congestion', '4. Ùn tắc', '#e74c3c'),
    ('tls_optimized', '5. Tối ưu đèn', '#9b59b6'),
    ('tls_actuated', '6. Đèn thông minh', '#1abc9c'),
]

summary_data = {}
tripinfo_data = {}
kpi_list = []

for key, label, color in SCENARIOS:
    sum_file = os.path.join(DATA_DIR, f'summary_{key}.xml')
    trip_file = os.path.join(DATA_DIR, f'tripinfo_{key}.xml')
    
    # Đọc summary_*.xml
    if os.path.exists(sum_file) and os.path.getsize(sum_file) > 100:
        with open(sum_file, 'r', encoding='utf-8') as f:
            content = f.read().strip()
        if not content.endswith('</summary>'):
            content += '\n</summary>'
        root = ET.fromstring(content)
        df_s = pd.DataFrame([s.attrib for s in root.findall('step')])
        for col in ['time', 'running', 'halting', 'meanSpeed', 'meanWaitingTime']:
            if col in df_s.columns:
                df_s[col] = pd.to_numeric(df_s[col], errors='coerce')
        df_s['meanSpeedKmh'] = df_s['meanSpeed'].apply(lambda x: x * 3.6 if x >= 0 else None)
        summary_data[key] = df_s
        
    # Đọc tripinfo_*.xml
    if os.path.exists(trip_file) and os.path.getsize(trip_file) > 100:
        with open(trip_file, 'r', encoding='utf-8') as f:
            content = f.read().strip()
        if not content.endswith('</tripinfos>'):
            content += '\n</tripinfos>'
        root = ET.fromstring(content)
        df_t = pd.DataFrame([t.attrib for t in root.findall('tripinfo')])
        for col in ['duration', 'routeLength', 'waitingTime', 'timeLoss']:
            if col in df_t.columns:
                df_t[col] = pd.to_numeric(df_t[col], errors='coerce')
        df_t['speedKmh'] = (df_t['routeLength'] / df_t['duration']) * 3.6
        tripinfo_data[key] = df_t
        
        kpi_list.append({
            'Kịch bản': label,
            'Số xe hoàn thành': len(df_t),
            'Vận tốc TB (km/h)': df_t['speedKmh'].mean(),
            'Thời gian chờ TB (s)': df_t['waitingTime'].mean(),
            'Tổn thất TG TB (s)': df_t['timeLoss'].mean()
        })

df_kpi = pd.DataFrame(kpi_list)
display(df_kpi)

## 3. Biểu đồ So sánh Vận tốc Trung bình (km/h) giữa các Kịch bản

In [ ]:
plt.figure(figsize=(10, 5))
bars = plt.bar(df_kpi['Kịch bản'], df_kpi['Vận tốc TB (km/h)'], color=['#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#1abc9c'], edgecolor='black')
plt.title('So sánh Vận tốc Trung bình của các Kịch bản (km/h)', fontsize=14, fontweight='bold')
plt.ylabel('Vận tốc (km/h)')
plt.xticks(rotation=20)
for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}', (bar.get_x() + bar.get_width()/2, h), xytext=(0, 4), textcoords='offset points', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Biểu đồ So sánh Thời gian Chờ (Waiting Time) & Tổn thất Thời gian (Time Loss)

In [ ]:
df_plot = df_kpi.melt(id_vars=['Kịch bản'], value_vars=['Thời gian chờ TB (s)', 'Tổn thất TG TB (s)'], var_name='Chỉ số', value_name='Thời gian (giây)')
plt.figure(figsize=(11, 5))
sns.barplot(data=df_plot, x='Kịch bản', y='Thời gian (giây)', hue='Chỉ số', palette=['#e74c3c', '#f39c12'])
plt.title('So sánh Thời gian Chờ và Tổn thất Thời gian do Ùn ứ (giây)', fontsize=14, fontweight='bold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. Diễn biến Số lượng Phương tiện trên Mạng lưới theo Thời gian

In [ ]:
plt.figure(figsize=(12, 6))
for key, label, color in SCENARIOS:
    if key in summary_data:
        df_s = summary_data[key].iloc[::5]
        plt.plot(df_s['time'], df_s['running'], label=label, color=color, linewidth=2)
plt.title('Diễn biến Số lượng Xe đang Lưu thông (Running Vehicles) theo Thời gian', fontsize=14, fontweight='bold')
plt.xlabel('Thời gian mô phỏng (giây)')
plt.ylabel('Số lượng phương tiện')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Phân bố Thời gian Chờ theo Loại Xe trong Giờ Cao điểm (Boxplot)

In [ ]:
if 'peak' in tripinfo_data:
    df_p = tripinfo_data['peak'].copy()
    vtype_map = {'moped': 'Xe máy (75%)', 'passenger': 'Ô tô con (18%)', 'bus': 'Xe buýt (4%)', 'truck': 'Xe tải (3%)'}
    df_p['Phân loại xe'] = df_p['vType'].map(vtype_map).fillna(df_p['vType'])
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df_p, x='Phân loại xe', y='waitingTime', palette={'Xe máy (75%)': '#e74c3c', 'Ô tô con (18%)': '#2ecc71', 'Xe buýt (4%)': '#3498db', 'Xe tải (3%)': '#f1c40f'})
    plt.title('Phân bố Thời gian Dừng chờ theo Loại Phương tiện (Giờ Cao điểm)', fontsize=14, fontweight='bold')
    plt.ylabel('Thời gian chờ (giây)')
    plt.tight_layout()
    plt.show()